# AML Feature Engineering -- Graph, Velocity, Balance & Fraud Intensity Score
---
Reads the rules-engine output and computes **90+ new features** across 5 categories:

| Category | Features | Description |
|----------|----------|-------------|
| Sender Account Velocity | 20 | Inflow/outflow counts & amounts at 1h, 24h, 7d, 30d |
| Sender Customer Velocity | 20 | Same metrics rolled up across all customer accounts |
| Sender Balance Tracking | 6 | Running balance, daily change, balance ratios |
| Receiver Account Velocity + Balance | 26 | Mirror of sender metrics for counterparty |
| IP Risk Score | 9 | VPN, emulator, night, FATF country, shared IP, geo-mismatch |
| Fraud Intensity Score (FIS) | 3 | Composite raw score, normalized 0-100, and band |

### Output
`stg_transactions_features.parquet` -- original data + all new columns



## 1 -- Environment Setup


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict
import warnings, os

warnings.filterwarnings('ignore')
OUTPUT_DIR = "../outputs_updated"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 -- Load Data


In [2]:
INPUT_FILE = "../outputs_updated/stg_transactions_rules.parquet"

if not os.path.exists(INPUT_FILE):
    for alt in ["stg_transactions_rules.parquet",
                "../outputs_updated/stg_transactions_rules.csv",
                "stg_transactions_rules.csv"]:
        if os.path.exists(alt):
            INPUT_FILE = alt
            break

ext = INPUT_FILE.rsplit('.', 1)[-1].lower()
df = pd.read_parquet(INPUT_FILE) if ext == 'parquet' else pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Loaded: {INPUT_FILE}")
print(f"  {len(df):,} rows x {len(df.columns)} columns")



Loaded: ../outputs_updated/stg_transactions_rules.parquet
  333,038 rows x 151 columns


## 3 -- Resolve Columns & Parse Datetime


In [3]:
def find_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

COL_ACCT     = find_col(["customer_account_number","account_number"])
COL_CP_ACCT  = find_col(["counterparty_account_number","cp_account_number"])
COL_CIF      = find_col(["customer_cif_id","cif_id"])
COL_AMOUNT   = find_col(["transaction_amount","amount"])
COL_TYPE     = find_col(["transaction_type_dr_cr","txn_type"])
COL_TS       = find_col(["timestamp","Timestamp"])
COL_DS       = find_col(["datestamp","Datestamp"])
COL_STATUS   = find_col(["transaction_status","txn_status"])
COL_RULE_SCORE = find_col(["rule_score"])
COL_VPN      = find_col(["vpn_flag"])
COL_EMULATOR = find_col(["emulator_flag"])
COL_IP       = find_col(["ip_address"])
COL_DEVICE   = find_col(["device_id_fingerprint","device_id"])

print("Key columns:")
for k, v in [("Account", COL_ACCT), ("Counterparty", COL_CP_ACCT), ("CIF", COL_CIF),
             ("Amount", COL_AMOUNT), ("Type", COL_TYPE), ("Rule Score", COL_RULE_SCORE)]:
    print(f"  {k:<15s} -> {str(v)}")

# Parse datetime
def parse_dt(row):
    ds = str(row.get(COL_DS, ""))
    ts = str(row.get(COL_TS, "00:00:00"))
    for fmt in ["%d-%m-%Y %H:%M:%S","%d/%m/%Y %H:%M:%S","%Y-%m-%d %H:%M:%S"]:
        try:
            return datetime.strptime(f"{ds} {ts}".strip(), fmt)
        except:
            continue
    return pd.NaT

df["_dt"] = df.apply(parse_dt, axis=1)
df["_amt"] = pd.to_numeric(df[COL_AMOUNT], errors="coerce").fillna(0)
df["_acct"] = df[COL_ACCT].astype(str).str.strip()
df["_cp"] = df[COL_CP_ACCT].astype(str).str.strip()
df["_cif"] = df[COL_CIF].astype(str).str.strip() if COL_CIF else df["_acct"]
df["_type"] = df[COL_TYPE].astype(str).str.strip().str.upper() if COL_TYPE else "DR"
df["_stat"] = df[COL_STATUS].astype(str).str.strip().str.upper() if COL_STATUS else "SUCCESS"
df["_date_only"] = df["_dt"].dt.date

# Determine direction: is this a debit (outflow) or credit (inflow) for the customer account
df["_is_debit"] = df["_type"].isin(["DR","DEBIT","D"])
df["_is_credit"] = df["_type"].isin(["CR","CREDIT","C"])

# Signed amount from sender perspective
df["_signed_amt"] = np.where(df["_is_debit"], -df["_amt"], df["_amt"])

print(f"Datetime parsed. Valid: {df['_dt'].notna().sum():,}")



Key columns:
  Account         -> customer_account_number
  Counterparty    -> counterparty_account_number
  CIF             -> customer_cif_id
  Amount          -> transaction_amount
  Type            -> transaction_type_dr_cr
  Rule Score      -> rule_score
Datetime parsed. Valid: 333,038


## 4 -- Sender Account Velocity Features (20 columns)
Rolling 1h, 24h, 7d, 30d windows: inflow/outflow counts and amounts.


In [4]:
print("Computing sender account velocity features...")
df = df.sort_values(["_acct", "_dt"]).reset_index(drop=True)

WINDOWS = {
    "1h":  timedelta(hours=1),
    "24h": timedelta(hours=24),
    "7d":  timedelta(days=7),
    "30d": timedelta(days=30),
}

# Initialize all sender account velocity columns
for w in WINDOWS:
    df[f"sender_acct_txn_count_{w}"] = 0
    df[f"sender_acct_inflow_amt_{w}"] = 0.0
    df[f"sender_acct_outflow_amt_{w}"] = 0.0
    df[f"sender_acct_inflow_count_{w}"] = 0
    df[f"sender_acct_outflow_count_{w}"] = 0

groups = df.groupby("_acct")
total_groups = len(groups)
processed = 0

for acct, gdf in groups:
    processed += 1
    if processed % 3000 == 0:
        print(f"  Sender acct: {processed:,}/{total_groups:,}...")

    idxs = gdf.index.values
    dts = gdf["_dt"].values
    amts = gdf["_amt"].values
    is_deb = gdf["_is_debit"].values
    is_cre = gdf["_is_credit"].values
    n = len(idxs)

    for wi, (wname, wdelta) in enumerate(WINDOWS.items()):
        wdelta_ns = wdelta / timedelta(seconds=1) * 1e9  # nanoseconds
        txn_cnt = np.zeros(n, dtype=int)
        in_amt = np.zeros(n, dtype=float)
        out_amt = np.zeros(n, dtype=float)
        in_cnt = np.zeros(n, dtype=int)
        out_cnt = np.zeros(n, dtype=int)

        left = 0
        for i in range(n):
            t = dts[i]
            if pd.isna(t):
                continue
            # Move left pointer
            while left < i and (t - dts[left]) / np.timedelta64(1, 'ns') > wdelta_ns:
                left += 1
            # Sum from left to i-1
            for j in range(left, i):
                if pd.isna(dts[j]):
                    continue
                txn_cnt[i] += 1
                if is_cre[j]:
                    in_amt[i] += amts[j]
                    in_cnt[i] += 1
                if is_deb[j]:
                    out_amt[i] += amts[j]
                    out_cnt[i] += 1

        df.loc[idxs, f"sender_acct_txn_count_{wname}"] = txn_cnt
        df.loc[idxs, f"sender_acct_inflow_amt_{wname}"] = np.round(in_amt, 2)
        df.loc[idxs, f"sender_acct_outflow_amt_{wname}"] = np.round(out_amt, 2)
        df.loc[idxs, f"sender_acct_inflow_count_{wname}"] = in_cnt
        df.loc[idxs, f"sender_acct_outflow_count_{wname}"] = out_cnt

print("  Sender account velocity: 20 columns done")



Computing sender account velocity features...
  Sender acct: 3,000/4,766...
  Sender account velocity: 20 columns done


## 5 -- Sender Customer Velocity Features (20 columns)
Same windows rolled up by customer CIF across all their accounts.


In [5]:
print("Computing sender customer velocity features...")
df = df.sort_values(["_cif", "_dt"]).reset_index(drop=True)

for w in WINDOWS:
    df[f"sender_cust_txn_count_{w}"] = 0
    df[f"sender_cust_inflow_amt_{w}"] = 0.0
    df[f"sender_cust_outflow_amt_{w}"] = 0.0
    df[f"sender_cust_inflow_count_{w}"] = 0
    df[f"sender_cust_outflow_count_{w}"] = 0

df["sender_cust_id_for_rollup"] = df["_cif"]

groups = df.groupby("_cif")
total_groups = len(groups)
processed = 0

for cif, gdf in groups:
    processed += 1
    if processed % 2000 == 0:
        print(f"  Sender cust: {processed:,}/{total_groups:,}...")

    idxs = gdf.index.values
    dts = gdf["_dt"].values
    amts = gdf["_amt"].values
    is_deb = gdf["_is_debit"].values
    is_cre = gdf["_is_credit"].values
    n = len(idxs)

    for wname, wdelta in WINDOWS.items():
        wdelta_ns = wdelta / timedelta(seconds=1) * 1e9
        txn_cnt = np.zeros(n, dtype=int)
        in_amt = np.zeros(n, dtype=float)
        out_amt = np.zeros(n, dtype=float)
        in_cnt = np.zeros(n, dtype=int)
        out_cnt = np.zeros(n, dtype=int)

        left = 0
        for i in range(n):
            t = dts[i]
            if pd.isna(t):
                continue
            while left < i and (t - dts[left]) / np.timedelta64(1, 'ns') > wdelta_ns:
                left += 1
            for j in range(left, i):
                if pd.isna(dts[j]):
                    continue
                txn_cnt[i] += 1
                if is_cre[j]:
                    in_amt[i] += amts[j]
                    in_cnt[i] += 1
                if is_deb[j]:
                    out_amt[i] += amts[j]
                    out_cnt[i] += 1

        df.loc[idxs, f"sender_cust_txn_count_{wname}"] = txn_cnt
        df.loc[idxs, f"sender_cust_inflow_amt_{wname}"] = np.round(in_amt, 2)
        df.loc[idxs, f"sender_cust_outflow_amt_{wname}"] = np.round(out_amt, 2)
        df.loc[idxs, f"sender_cust_inflow_count_{wname}"] = in_cnt
        df.loc[idxs, f"sender_cust_outflow_count_{wname}"] = out_cnt

print("  Sender customer velocity: 20 columns done")



Computing sender customer velocity features...
  Sender cust: 2,000/3,291...
  Sender customer velocity: 20 columns done


## 6 -- Sender Balance Tracking (6 columns)
Running balance, daily cumulative change, and balance ratios.


In [6]:
print("Computing sender balance tracking...")
df = df.sort_values(["_acct", "_dt"]).reset_index(drop=True)

df["sender_balance_before_txn"] = 0.0
df["sender_running_balance_txn_amount"] = df["_signed_amt"]
df["sender_balance_after_txn"] = 0.0
df["sender_cumulative_daily_balance_change"] = 0.0
df["sender_current_balance"] = 0.0
df["sender_bal_ratio_after_to_current"] = 0.0

for acct, gdf in df.groupby("_acct"):
    idxs = gdf.index.values
    signed = gdf["_signed_amt"].values
    dates = gdf["_date_only"].values
    n = len(idxs)

    bal = 0.0
    daily_change = 0.0
    prev_date = None
    bal_before = np.zeros(n)
    bal_after = np.zeros(n)
    daily_cum = np.zeros(n)

    for i in range(n):
        curr_date = dates[i]
        if prev_date is not None and curr_date != prev_date:
            daily_change = 0.0
        prev_date = curr_date

        bal_before[i] = bal
        bal += signed[i]
        bal_after[i] = bal
        daily_change += signed[i]
        daily_cum[i] = daily_change

    df.loc[idxs, "sender_balance_before_txn"] = np.round(bal_before, 2)
    df.loc[idxs, "sender_balance_after_txn"] = np.round(bal_after, 2)
    df.loc[idxs, "sender_cumulative_daily_balance_change"] = np.round(daily_cum, 2)
    df.loc[idxs, "sender_current_balance"] = round(bal, 2)

# Balance ratio
df["sender_bal_ratio_after_to_current"] = np.where(
    df["sender_current_balance"] != 0,
    np.round(df["sender_balance_after_txn"] / df["sender_current_balance"], 4),
    0.0
)

print("  Sender balance: 6 columns done")



Computing sender balance tracking...
  Sender balance: 6 columns done


## 7 -- Receiver Account Velocity + Balance (26 columns)
Mirror of sender metrics computed from the counterparty's perspective.
For external/unknown counterparties, values default to 0.


In [7]:
print("Computing receiver account velocity + balance features...")

# Build receiver-centric lookup: for each account, pre-compute its velocity at each timestamp
# We already computed sender metrics. For receiver, we join sender metrics on counterparty account.

# First, build a lookup: acct -> sorted list of (dt, inflow_amt, outflow_amt, inflow_cnt, outflow_cnt, balance)
# from the sender computation (which covers ALL accounts as senders)

# Efficient approach: create a mapping from acct -> its row indices, then for each transaction,
# look up the counterparty's metrics at the closest preceding timestamp.

print("  Building receiver lookup tables...")

# Pre-build per-account sorted arrays for binary search
acct_data = {}
for acct, gdf in df.groupby("_acct"):
    gdf_sorted = gdf.sort_values("_dt")
    acct_data[acct] = {
        "dts": gdf_sorted["_dt"].values,
        "idxs": gdf_sorted.index.values,
    }

# Initialize receiver columns
for w in WINDOWS:
    df[f"receiver_acct_txn_count_{w}"] = 0
    df[f"receiver_acct_inflow_amt_{w}"] = 0.0
    df[f"receiver_acct_outflow_amt_{w}"] = 0.0
    df[f"receiver_acct_inflow_count_{w}"] = 0
    df[f"receiver_acct_outflow_count_{w}"] = 0

df["receiver_balance_before_txn"] = 0.0
df["receiver_running_balance_txn_amount"] = 0.0
df["receiver_balance_after_txn"] = 0.0
df["receiver_cumulative_daily_balance_change"] = 0.0
df["receiver_current_balance"] = 0.0
df["receiver_bal_ratio_after_to_current"] = 0.0

# For each transaction, look up the counterparty's sender metrics
# (since counterparty is also a sender in some other transaction)
print("  Mapping receiver metrics from counterparty's sender data...")

# Build fast lookup: for each account, its latest computed sender metrics
# We map receiver columns from the sender columns of the counterparty account
# at the closest prior timestamp using the pre-computed sender_acct_* columns

# Group by counterparty, and for each row, find the counterparty's own metrics
cp_groups = df.groupby("_cp")
total_cp = len(cp_groups)
processed = 0

for cp_acct, gdf_main in cp_groups:
    processed += 1
    if processed % 5000 == 0:
        print(f"    Receiver mapping: {processed:,}/{total_cp:,}...")

    if cp_acct in ("", "nan", "None"):
        continue

    # Get the counterparty's own transactions (where it is the sender/_acct)
    if cp_acct not in acct_data:
        continue

    cp_own = df.loc[df["_acct"] == cp_acct].sort_values("_dt")
    if len(cp_own) == 0:
        continue

    cp_dts = cp_own["_dt"].values
    cp_idxs_arr = cp_own.index.values

    # For each row where this account is the counterparty, find nearest prior row in cp_own
    main_idxs = gdf_main.index.values
    main_dts = gdf_main["_dt"].values

    for i in range(len(main_idxs)):
        t = main_dts[i]
        if pd.isna(t):
            continue
        # Binary search for latest cp transaction before t
        pos = np.searchsorted(cp_dts, t, side='right') - 1
        if pos < 0:
            continue
        cp_idx = cp_idxs_arr[pos]

        # Copy sender metrics from counterparty's row as receiver metrics
        row_idx = main_idxs[i]
        for w in WINDOWS:
            df.at[row_idx, f"receiver_acct_txn_count_{w}"] = df.at[cp_idx, f"sender_acct_txn_count_{w}"]
            df.at[row_idx, f"receiver_acct_inflow_amt_{w}"] = df.at[cp_idx, f"sender_acct_inflow_amt_{w}"]
            df.at[row_idx, f"receiver_acct_outflow_amt_{w}"] = df.at[cp_idx, f"sender_acct_outflow_amt_{w}"]
            df.at[row_idx, f"receiver_acct_inflow_count_{w}"] = df.at[cp_idx, f"sender_acct_inflow_count_{w}"]
            df.at[row_idx, f"receiver_acct_outflow_count_{w}"] = df.at[cp_idx, f"sender_acct_outflow_count_{w}"]

        df.at[row_idx, "receiver_balance_before_txn"] = df.at[cp_idx, "sender_balance_before_txn"]
        df.at[row_idx, "receiver_running_balance_txn_amount"] = df.at[cp_idx, "sender_running_balance_txn_amount"]
        df.at[row_idx, "receiver_balance_after_txn"] = df.at[cp_idx, "sender_balance_after_txn"]
        df.at[row_idx, "receiver_cumulative_daily_balance_change"] = df.at[cp_idx, "sender_cumulative_daily_balance_change"]
        df.at[row_idx, "receiver_current_balance"] = df.at[cp_idx, "sender_current_balance"]

# Receiver balance ratio
df["receiver_bal_ratio_after_to_current"] = np.where(
    df["receiver_current_balance"] != 0,
    np.round(df["receiver_balance_after_txn"] / df["receiver_current_balance"], 4),
    0.0
)

# receiver_account_outflow_30d (reuse)
df["receiver_account_outflow_30d"] = df["receiver_acct_outflow_count_30d"]

print("  Receiver velocity + balance: 26 columns done")



Computing receiver account velocity + balance features...
  Building receiver lookup tables...
  Mapping receiver metrics from counterparty's sender data...
  Receiver velocity + balance: 26 columns done


## 8 -- Volume Balance Ratios (2 columns)
Measures how much of inflows are immediately re-forwarded.


In [8]:
print("Computing volume balance ratios...")

# 24h ratio
s_in_24 = df["sender_acct_inflow_amt_24h"]
s_out_24 = df["sender_acct_outflow_amt_24h"]
df["inflow_outflow_volume_balance_ratio_24h"] = np.where(
    s_in_24 > 0,
    np.round(np.minimum(s_in_24, s_out_24) / s_in_24, 4),
    0.0
)

# 7d ratio
s_in_7d = df["sender_acct_inflow_amt_7d"]
s_out_7d = df["sender_acct_outflow_amt_7d"]
df["inflow_outflow_volume_balance_ratio_7d"] = np.where(
    s_in_7d > 0,
    np.round(np.minimum(s_in_7d, s_out_7d) / s_in_7d, 4),
    0.0
)

print("  Volume balance ratios: 2 columns done")



Computing volume balance ratios...
  Volume balance ratios: 2 columns done


## 9 -- IP Risk Score (9 columns)
Composite IP-level risk signal combining VPN, emulator, night hours, FATF
high-risk country, cross-border, shared IP across accounts, geo-mismatch
between IP location and registered address, minus KYC verification credit.

**Formula**: `score = 0.10 (base) + vpn(0.15) + emulator(0.10) + night(0.10)`
`+ country_high(0.10) + cross_border(0.05) + shared_ip(0.10) + geo_mismatch(0.10)`
`- kyc_verified(0.10)` clipped to [0.0, 1.0]


In [9]:
print("Computing IP risk score...")

# -- Resolve flag columns --
def flag_series(col_name):
    if col_name and col_name in df.columns:
        return df[col_name].astype(str).str.strip().str.upper().isin(["Y","1","TRUE","YES"])
    return pd.Series(False, index=df.index)

COL_VPN      = find_col(["vpn_flag", "VPN Flag"])
COL_EMULATOR = find_col(["emulator_flag", "Emulator Flag"])
COL_IP       = find_col(["ip_address", "IP Address of Originating Device"])
COL_GEO      = find_col(["geo_location_city_country", "geo_location", "Geo-Location (City/Country)"])
COL_SENDER   = find_col(["sender_country_code", "sender_country", "Sender Country Code*"])
COL_RECEIVER = find_col(["receiver_country_code", "receiver_country", "Receiver Country Code*"])
COL_RISK     = find_col(["customer_current_risk_score", "risk_score", "Customer Current Risk Score"])
COL_WALKYC   = find_col(["wallet_kyc_category", "wallet_kyc", "Wallet KYC Category"])
COL_VKYC     = find_col(["vkyc_flag", "VKYC Flag"])

is_vpn      = flag_series(COL_VPN)
is_emulator = flag_series(COL_EMULATOR)
is_vkyc     = flag_series(COL_VKYC)

# -- Night transaction (22:00 - 06:00) --
hour = df["_dt"].dt.hour if "_dt" in df.columns else pd.Series(12, index=df.index)
is_night = (hour >= 22) | (hour < 6)

# -- High-risk country (FATF grey/blacklist) --
FATF_HIGH_RISK = {"KP","IR","MM","AF","SY","YE","HT","SD","ML","BF","MZ","TZ",
                   "CD","SS","LY","PK","NI","JM","TR","PH"}

sender_high = df[COL_SENDER].astype(str).str.strip().str.upper().isin(FATF_HIGH_RISK) if COL_SENDER and COL_SENDER in df.columns else pd.Series(False, index=df.index)
receiver_high = df[COL_RECEIVER].astype(str).str.strip().str.upper().isin(FATF_HIGH_RISK) if COL_RECEIVER and COL_RECEIVER in df.columns else pd.Series(False, index=df.index)
country_high = sender_high | receiver_high

# -- Cross-border (sender != receiver country) --
is_cross_border = pd.Series(False, index=df.index)
if COL_SENDER and COL_RECEIVER and COL_SENDER in df.columns and COL_RECEIVER in df.columns:
    s_cc = df[COL_SENDER].astype(str).str.strip().str.upper()
    r_cc = df[COL_RECEIVER].astype(str).str.strip().str.upper()
    is_cross_border = (s_cc != r_cc) & (s_cc != "") & (r_cc != "") & (s_cc != "NAN") & (r_cc != "NAN")

# -- KYC strength (verified KYC = lower risk) --
kyc_verified = is_vkyc.copy()
if COL_WALKYC and COL_WALKYC in df.columns:
    full_kyc = df[COL_WALKYC].astype(str).str.lower().str.contains("full", na=False)
    kyc_verified = kyc_verified | full_kyc
if COL_RISK and COL_RISK in df.columns:
    low_risk = df[COL_RISK].astype(str).str.strip().str.upper().isin(["LOW","1"])
    kyc_verified = kyc_verified | low_risk

# -- IP frequency anomaly (same IP used by multiple distinct accounts) --
ip_shared_score = pd.Series(0.0, index=df.index)
if COL_IP and COL_IP in df.columns:
    ip_unique_accts = df.groupby(df[COL_IP].astype(str).str.strip())[COL_ACCT].transform("nunique")
    ip_acct_count = ip_unique_accts.fillna(1)
    # Normalize: 1 account = 0, 2 = 0.25, 3 = 0.50, 5+ = 1.0
    ip_shared_score = np.clip((ip_acct_count - 1) / 4, 0, 1)
else:
    ip_acct_count = pd.Series(1, index=df.index)

# -- Geo mismatch (IP GPS location far from registered address) --
geo_mismatch = pd.Series(False, index=df.index)
gps_lat_col = find_col(["gps_coordinates_lat"])
cust_lat_col = find_col(["customer_address_lat"])
gps_lon_col = find_col(["gps_coordinates_lon"])
cust_lon_col = find_col(["customer_address_lon"])

if (gps_lat_col and cust_lat_col and gps_lon_col and cust_lon_col
    and gps_lat_col in df.columns and cust_lat_col in df.columns
    and gps_lon_col in df.columns and cust_lon_col in df.columns):
    glat = pd.to_numeric(df[gps_lat_col], errors="coerce").fillna(0)
    clat = pd.to_numeric(df[cust_lat_col], errors="coerce").fillna(0)
    glon = pd.to_numeric(df[gps_lon_col], errors="coerce").fillna(0)
    clon = pd.to_numeric(df[cust_lon_col], errors="coerce").fillna(0)
    # Distance in degrees (1 degree ~ 111 km); flag if > 2 degrees (~220 km)
    dist_deg = np.sqrt((glat - clat)**2 + (glon - clon)**2)
    geo_mismatch = (dist_deg > 2.0) & (glat != 0) & (clat != 0)

# ── Compute IP Score ──
ip_score = (
    0.10                                        # base score
    + is_vpn.astype(float)          * 0.15      # VPN active
    + is_emulator.astype(float)     * 0.10      # emulator detected
    + is_night.astype(float)        * 0.10      # night transaction
    + country_high.astype(float)    * 0.10      # FATF high-risk country
    + is_cross_border.astype(float) * 0.05      # cross-border transaction
    + ip_shared_score               * 0.10      # shared IP across accounts
    + geo_mismatch.astype(float)    * 0.10      # IP location far from address
    - kyc_verified.astype(float)    * 0.10      # KYC verified reduces risk
)

df["ip_risk_score"] = np.clip(np.round(ip_score, 4), 0.0, 1.0)

# Component flags for explainability
df["ip_flag_vpn"] = is_vpn.astype(int)
df["ip_flag_emulator"] = is_emulator.astype(int)
df["ip_flag_night"] = is_night.astype(int)
df["ip_flag_country_high_risk"] = country_high.astype(int)
df["ip_flag_cross_border"] = is_cross_border.astype(int)
df["ip_flag_shared_ip"] = (ip_acct_count > 1).astype(int) if COL_IP and COL_IP in df.columns else 0
df["ip_flag_geo_mismatch"] = geo_mismatch.astype(int)
df["ip_flag_kyc_verified"] = kyc_verified.astype(int)

# Summary
print(f"\nIP Risk Score Distribution:")
print(f"  Min:    {df['ip_risk_score'].min():.3f}")
print(f"  Median: {df['ip_risk_score'].median():.3f}")
print(f"  Mean:   {df['ip_risk_score'].mean():.3f}")
print(f"  P95:    {df['ip_risk_score'].quantile(0.95):.3f}")
print(f"  Max:    {df['ip_risk_score'].max():.3f}")

print(f"\nFlag trigger rates:")
for flag_col in [c for c in df.columns if c.startswith("ip_flag_")]:
    rate = df[flag_col].mean() * 100
    print(f"  {flag_col:<30s} {rate:>6.2f}%")

print(f"\n  IP risk score: 9 columns done (1 score + 8 component flags)")



Computing IP risk score...

IP Risk Score Distribution:
  Min:    0.000
  Median: 0.025
  Mean:   0.065
  P95:    0.175
  Max:    0.425

Flag trigger rates:
  ip_flag_vpn                      2.75%
  ip_flag_emulator                 1.00%
  ip_flag_night                    9.99%
  ip_flag_country_high_risk        0.40%
  ip_flag_cross_border             3.52%
  ip_flag_shared_ip               62.34%
  ip_flag_geo_mismatch             0.00%
  ip_flag_kyc_verified            67.82%

  IP risk score: 9 columns done (1 score + 8 component flags)


## 10 -- Fraud Intensity Score (3 columns)
Composite risk score using rule_score, behavioural velocity, ip_risk_score, and device signals.

**Formula**: `raw = rule_risk x 35 + behaviour_risk x 30 + ip_risk x 18 + device_risk x 17`
**Normalized**: `FIS = min(raw / p99, 1) x 100`
**Bands**: very_low (0-20), low (20-40), medium (40-60), high (60-80), critical (80-100)


In [10]:
print("Computing Fraud Intensity Score...")

# ── Component 1: Rule Risk (from rule_score) ──
rule_score_col = find_col(["rule_score"])
rule_score = pd.to_numeric(df.get(rule_score_col, 0), errors="coerce").fillna(0) if rule_score_col else pd.Series(0, index=df.index)
rule_p99 = max(rule_score.quantile(0.99), 1)
rule_risk = np.clip(rule_score / rule_p99, 0, 1)

# ── Component 2: Behaviour Risk (velocity + volume balance ratio) ──
behaviour_risk = np.clip(
    np.clip(df["sender_acct_txn_count_1h"] / 5, 0, 1) * 0.3 +
    np.clip(df["sender_acct_txn_count_24h"] / 20, 0, 1) * 0.2 +
    np.clip(df["inflow_outflow_volume_balance_ratio_24h"], 0, 1) * 0.3 +
    np.clip(df["sender_acct_outflow_amt_24h"] / 500000, 0, 1) * 0.2,
    0, 1
)

# ── Component 3: IP Risk (pre-computed in previous cell) ──
ip_risk = df["ip_risk_score"]

# ── Component 4: Device Risk ──
emu_flag = df["ip_flag_emulator"] if "ip_flag_emulator" in df.columns else 0
vpn_flag = df["ip_flag_vpn"] if "ip_flag_vpn" in df.columns else 0
device_risk = np.clip(
    emu_flag * 0.7 +
    vpn_flag * 0.3,
    0, 1
)

# ── Composite FIS ──
fis_raw = (
    rule_risk       * 35 +
    behaviour_risk  * 30 +
    ip_risk         * 18 +
    device_risk     * 17
)

df["fraud_intensity_score_raw"] = np.round(fis_raw, 4)

# Normalize to 0-100
fis_p99 = max(fis_raw.quantile(0.99), 0.01)
fis_normalized = np.clip(fis_raw / fis_p99, 0, 1) * 100
df["fraud_intensity_score"] = np.round(fis_normalized, 2)

# Bands
def fis_band(score):
    if score >= 80: return "critical"
    if score >= 60: return "high"
    if score >= 40: return "medium"
    if score >= 20: return "low"
    return "very_low"

df["fis_band"] = df["fraud_intensity_score"].apply(fis_band)

# Summary
print(f"\nFIS Distribution:")
print(f"  {'Band':<12s} {'Count':>10s} {'%':>7s}")
print(f"  {'-'*32}")
for band in ["critical","high","medium","low","very_low"]:
    cnt = (df["fis_band"] == band).sum()
    print(f"  {band:<12s} {cnt:>10,} {cnt/len(df)*100:>6.2f}%")

print(f"\n  Component weights: rule=35, behaviour=30, ip=18, device=17")
print(f"\n  Raw score:  min={fis_raw.min():.2f}  median={fis_raw.median():.2f}  "
      f"mean={fis_raw.mean():.2f}  p99={fis_p99:.2f}  max={fis_raw.max():.2f}")
print(f"  Normalized: min={df['fraud_intensity_score'].min():.1f}  "
      f"median={df['fraud_intensity_score'].median():.1f}  "
      f"max={df['fraud_intensity_score'].max():.1f}")

print(f"\n  Fraud Intensity Score: 3 columns done")



Computing Fraud Intensity Score...

FIS Distribution:
  Band              Count       %
  --------------------------------
  critical         14,500   4.35%
  high             22,081   6.63%
  medium           42,109  12.64%
  low              86,377  25.94%
  very_low        167,971  50.44%

  Component weights: rule=35, behaviour=30, ip=18, device=17

  Raw score:  min=0.00  median=7.93  mean=10.10  p99=39.69  max=66.29
  Normalized: min=0.0  median=20.0  max=100.0

  Fraud Intensity Score: 3 columns done


## 12 -- Summary & Export


In [11]:
# Drop internal working columns
internal = [c for c in df.columns if c.startswith("_")]
df_out = df.drop(columns=internal)

# Count new features
original_cols = set()
for alt in ["stg_transactions_rules.parquet","stg_transactions_rules.csv"]:
    try:
        if alt.endswith('.parquet'):
            sample = pd.read_parquet(INPUT_FILE, columns=None)
        else:
            sample = pd.read_csv(INPUT_FILE, nrows=1)
        original_cols = set(sample.columns)
        break
    except:
        pass

new_cols = [c for c in df_out.columns if c not in original_cols and not c.startswith("_")]
print(f"New features added: {len(new_cols)}")
print(f"Total columns: {len(df_out.columns)}")

# Feature category breakdown
categories = {
    "Sender Acct Velocity": [c for c in new_cols if c.startswith("sender_acct_")],
    "Sender Cust Velocity": [c for c in new_cols if c.startswith("sender_cust_")],
    "Sender Balance":       [c for c in new_cols if c.startswith("sender_bal") or c.startswith("sender_balance") or c.startswith("sender_running") or c.startswith("sender_cumulative") or c.startswith("sender_current")],
    "Receiver Features":    [c for c in new_cols if c.startswith("receiver_")],
    "Volume Ratios":        [c for c in new_cols if "volume_balance_ratio" in c],
    "Fraud Intensity":      [c for c in new_cols if c.startswith("fraud_intensity") or c.startswith("fis_")],
}

print(f"\nFeature breakdown:")
for cat, cols in categories.items():
    print(f"  {cat:<25s} {len(cols):>3} columns")

# Export
path_out = os.path.join(OUTPUT_DIR, "stg_transactions_features.parquet")
df_out.to_parquet(path_out, index=False, compression='snappy')
size_mb = os.path.getsize(path_out) / (1024*1024)
print(f"\nExported: {path_out}")
print(f"  {len(df_out):,} rows x {len(df_out.columns)} columns")
print(f"  Size: {size_mb:.2f} MB")

print(f"\n{'='*60}")
print(f"FEATURE ENGINEERING COMPLETE")
print(f"Output: {os.path.abspath(path_out)}")
print(f"{'='*60}")



New features added: 88
Total columns: 239

Feature breakdown:
  Sender Acct Velocity       20 columns
  Sender Cust Velocity       21 columns
  Sender Balance              6 columns
  Receiver Features          27 columns
  Volume Ratios               2 columns
  Fraud Intensity             3 columns

Exported: ../outputs_updated\stg_transactions_features.parquet
  333,038 rows x 239 columns
  Size: 79.69 MB

FEATURE ENGINEERING COMPLETE
Output: c:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\outputs_updated\stg_transactions_features.parquet
